# Telco Customer Churn — 특성 공학 파이프라인
## STEP 01. 데이터 준비

**연구 주제:** 특성 공학 전략에 따른 통신사 고객 이탈 예측 성능 비교

- 데이터 출처: [Kaggle Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)
- 문제 유형: 이진 분류 (Classification)
- 타겟 변수: `Churn` (Yes=1, No=0)
- 원본 데이터: 7,043개 샘플, 21개 컬럼(식별자 1, 설명 변수 19, 타겟 1)

> STEP 01에서는 원본 데이터를 그대로 보존하고, 결측치 대체·인코딩·스케일링은 하지 않는다. 이후 실험의 공정성을 위해 실제 전처리는 학습 데이터에 적합한 파이프라인 안에서 수행한다.

## 1. 라이브러리 준비

In [ ]:
# 분석에 필요한 패키지 설치 명령
# %pip install pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
print('pandas version:', pd.__version__)

## 2. 데이터 로드
로컬 파일이 있다면 `data/WA_Fn-UseC_-Telco-Customer-Churn.csv`를 우선 사용한다. 없으면 공개 IBM 예제 CSV를 불러온다.

In [ ]:
local_path = Path('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
source_url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'

if local_path.exists():
    df = pd.read_csv(local_path, dtype={'TotalCharges': 'string'})
    print('로컬 파일에서 불러왔습니다.')
else:
    df = pd.read_csv(source_url, dtype={'TotalCharges': 'string'})
    print('IBM 공개 데이터 URL에서 불러왔습니다.')

print('데이터 shape:', df.shape)
display(df.head())

## 3. 기본 구조와 데이터 품질 확인

In [ ]:
print('컬럼 목록:')
print(df.columns.tolist())
print('\n자료형 및 결측치 요약:')
df.info()

print('\n중복 행:', df.duplicated().sum())
print('중복 customerID:', df['customerID'].duplicated().sum())
print('\n타겟 분포:')
display(df['Churn'].value_counts(dropna=False).rename_axis('Churn').to_frame('count'))
display((df['Churn'].value_counts(normalize=True) * 100).round(2).rename('percent'))

## 4. 숫자 형태로 저장된 문자열 확인
`TotalCharges`의 공백 문자열은 `isna()` 검사만으로 탐지되지 않을 수 있다. 원본 `df`를 유지한 검사용 시리즈에서 수치 변환 여부를 확인한다.

In [ ]:
total_charges_numeric = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')
blank_or_invalid = total_charges_numeric.isna()

print('원본 NA 개수:')
display(df.isna().sum()[lambda s: s > 0])
print('TotalCharges 숫자 변환 불가 개수:', int(blank_or_invalid.sum()))
display(df.loc[blank_or_invalid, ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])

## 5. 타겟 변수 정의 및 입력 변수 구분
타겟만 0과 1로 바꾸고, 설명 변수는 원본 상태로 보관한다. `customerID`는 예측에 적절하지 않은 식별자이므로 제외한다.

In [ ]:
y = df['Churn'].map({'No': 0, 'Yes': 1})
assert y.notna().all(), '예상치 못한 타겟 값이 있습니다.'
y = y.astype(int)
X = df.drop(columns=['Churn', 'customerID']).copy()

# SeniorCitizen은 고령 고객 여부를 나타내는 이진 범주형이다.
categorical_columns = X.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
categorical_columns.append('SeniorCitizen')
numeric_columns = [c for c in X.columns if c not in categorical_columns]

print('입력 데이터:', X.shape)
print('타겟 데이터:', y.shape)
print('수치형 후보:', numeric_columns)
print('범주형:', categorical_columns)
print('※ TotalCharges는 원본에서 문자열이므로 이후 수치형 변환 대상입니다.')

## 6. 컬럼 설명 표

In [ ]:
descriptions = {
'customerID':'고객 고유 식별자', 'gender':'성별', 'SeniorCitizen':'고령 고객 여부(0/1)',
'Partner':'배우자·파트너 유무', 'Dependents':'부양가족 유무', 'tenure':'가입 유지 기간(개월)',
'PhoneService':'전화 서비스 이용 여부', 'MultipleLines':'복수 회선 이용 여부',
'InternetService':'인터넷 서비스 유형', 'OnlineSecurity':'온라인 보안 서비스 이용 여부',
'OnlineBackup':'온라인 백업 서비스 이용 여부', 'DeviceProtection':'기기 보호 서비스 이용 여부',
'TechSupport':'기술 지원 서비스 이용 여부', 'StreamingTV':'TV 스트리밍 이용 여부',
'StreamingMovies':'영화 스트리밍 이용 여부', 'Contract':'계약 유형',
'PaperlessBilling':'전자 청구서 이용 여부', 'PaymentMethod':'결제 수단',
'MonthlyCharges':'월 청구 요금', 'TotalCharges':'누적 청구 요금', 'Churn':'고객 이탈 여부(타겟)'
}
column_table = pd.DataFrame({
    '컬럼명': df.columns,
    '자료형': [str(df[c].dtype) for c in df.columns],
    '설명': [descriptions.get(c, '') for c in df.columns]
})
display(column_table)

## STEP 01 정리

- 데이터 원본 크기와 컬럼 확인
- 이진 분류 타겟 `Churn` 정의
- 원본에 숨어 있는 `TotalCharges` 공백 탐지
- 식별자 `customerID` 제외
- 다음 단계에서는 Histogram, Boxplot, Heatmap, Countplot을 활용한 EDA를 수행

**주의:** 향후 결측치 처리·스케일링·변수 선택을 비교할 때는 학습/테스트 분리를 먼저 하고 학습 데이터에서만 전처리 파라미터를 학습해야 데이터 누수를 방지한다.